# 03 - Feature Engineering

This notebook turns the cleaned NFL data into features that can be used throughout the rest of the projection model.

The goal is to describe how teams performed, how their rosters were built, and what changed from one season to the next. These features will later be used for player projections, team strength ratings, and game predictions.

In [1]:
from pathlib import Path

import numpy as np
import polars as pl

PROJECT_ROOT = Path.cwd().parent
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

In [2]:
schedule = pl.read_parquet(
    PROCESSED_DIR / "schedule_clean.parquet"
)

print(schedule.shape)
schedule.head()

(2895, 16)


game_id,season,week,gameday,weekday,gametime,away_team,away_score,home_team,home_score,location,result,total,overtime,away_rest,home_rest
str,i32,i32,str,str,str,str,i32,str,i32,str,i32,i32,i32,i32,i32
"""2015_01_PIT_NE""",2015,1,"""2015-09-10""","""Thursday""","""20:30""","""PIT""",21,"""NE""",28,"""Home""",7,49,0,7,7
"""2015_01_BAL_DEN""",2015,1,"""2015-09-13""","""Sunday""","""16:25""","""BAL""",13,"""DEN""",19,"""Home""",6,32,0,7,7
"""2015_01_CAR_JAX""",2015,1,"""2015-09-13""","""Sunday""","""13:00""","""CAR""",20,"""JAX""",9,"""Home""",-11,29,0,7,7
"""2015_01_CIN_OAK""",2015,1,"""2015-09-13""","""Sunday""","""16:25""","""CIN""",33,"""LV""",13,"""Home""",-20,46,0,7,7
"""2015_01_CLE_NYJ""",2015,1,"""2015-09-13""","""Sunday""","""13:00""","""CLE""",10,"""NYJ""",31,"""Home""",21,41,0,7,7


# Team Season Performance

I want to start with one row for each team and season.

The schedule data gives us the basic results for every game, which lets me build features like wins, losses, points scored, points allowed, and point differential. This will become the base team-season table that other features can be added to later.

In [3]:
home_games = schedule.select([
    "game_id",
    "season",
    "week",
    pl.col("home_team").alias("team"),
    pl.col("away_team").alias("opponent"),
    pl.col("home_score").alias("points_for"),
    pl.col("away_score").alias("points_against"),
    pl.lit(1).alias("home_game")
])

away_games = schedule.select([
    "game_id",
    "season",
    "week",
    pl.col("away_team").alias("team"),
    pl.col("home_team").alias("opponent"),
    pl.col("away_score").alias("points_for"),
    pl.col("home_score").alias("points_against"),
    pl.lit(0).alias("home_game")
])

team_games = (
    pl.concat([
        home_games,
        away_games
    ])
    .with_columns(
        (pl.col("points_for") - pl.col("points_against"))
        .alias("point_diff")
    )
    .sort([
        "season",
        "week",
        "team"
    ])
)

team_games.head(10)

game_id,season,week,team,opponent,points_for,points_against,home_game,point_diff
str,i32,i32,str,str,i32,i32,i32,i32
"""2015_01_NO_ARI""",2015,1,"""ARI""","""NO""",31,19,1,12
"""2015_01_PHI_ATL""",2015,1,"""ATL""","""PHI""",26,24,1,2
"""2015_01_BAL_DEN""",2015,1,"""BAL""","""DEN""",13,19,0,-6
"""2015_01_IND_BUF""",2015,1,"""BUF""","""IND""",27,14,1,13
"""2015_01_CAR_JAX""",2015,1,"""CAR""","""JAX""",20,9,0,11
"""2015_01_GB_CHI""",2015,1,"""CHI""","""GB""",23,31,1,-8
"""2015_01_CIN_OAK""",2015,1,"""CIN""","""LV""",33,13,0,20
"""2015_01_CLE_NYJ""",2015,1,"""CLE""","""NYJ""",10,31,0,-21
"""2015_01_NYG_DAL""",2015,1,"""DAL""","""NYG""",27,26,1,1


In [4]:
team_games = team_games.with_columns([
    (pl.col("points_for") > pl.col("points_against"))
    .cast(pl.Int8)
    .alias("win"),

    (pl.col("points_for") < pl.col("points_against"))
    .cast(pl.Int8)
    .alias("loss"),

    (pl.col("points_for") == pl.col("points_against"))
    .cast(pl.Int8)
    .alias("tie")
])

team_games.head(10)

game_id,season,week,team,opponent,points_for,points_against,home_game,point_diff,win,loss,tie
str,i32,i32,str,str,i32,i32,i32,i32,i8,i8,i8
"""2015_01_NO_ARI""",2015,1,"""ARI""","""NO""",31,19,1,12,1,0,0
"""2015_01_PHI_ATL""",2015,1,"""ATL""","""PHI""",26,24,1,2,1,0,0
"""2015_01_BAL_DEN""",2015,1,"""BAL""","""DEN""",13,19,0,-6,0,1,0
"""2015_01_IND_BUF""",2015,1,"""BUF""","""IND""",27,14,1,13,1,0,0
"""2015_01_CAR_JAX""",2015,1,"""CAR""","""JAX""",20,9,0,11,1,0,0
"""2015_01_GB_CHI""",2015,1,"""CHI""","""GB""",23,31,1,-8,0,1,0
"""2015_01_CIN_OAK""",2015,1,"""CIN""","""LV""",33,13,0,20,1,0,0
"""2015_01_CLE_NYJ""",2015,1,"""CLE""","""NYJ""",10,31,0,-21,0,1,0
"""2015_01_NYG_DAL""",2015,1,"""DAL""","""NYG""",27,26,1,1,1,0,0


In [6]:
team_season = (
    team_games
    .group_by([
        "season",
        "team"
    ])
    .agg([
        pl.len().alias("games"),
        pl.col("win").sum().alias("wins"),
        pl.col("loss").sum().alias("losses"),
        pl.col("tie").sum().alias("ties"),
        pl.col("points_for").sum().alias("points_for"),
        pl.col("points_against").sum().alias("points_against"),
        pl.col("point_diff").sum().alias("point_diff")
    ])
    .with_columns([
        (
            (pl.col("wins") + 0.5 * pl.col("ties")) /
            pl.col("games")
        ).alias("win_pct"),

        (
            pl.col("points_for") /
            pl.col("games")
        ).alias("points_for_per_game"),

        (
            pl.col("points_against") /
            pl.col("games")
        ).alias("points_against_per_game"),

        (
            pl.col("point_diff") /
            pl.col("games")
        ).alias("point_diff_per_game")
    ])
    .sort([
        "season",
        "team"
    ])
)

team_season.head(10)

season,team,games,wins,losses,ties,points_for,points_against,point_diff,win_pct,points_for_per_game,points_against_per_game,point_diff_per_game
i32,str,u32,i64,i64,i64,i32,i32,i32,f64,f64,f64,f64
2015,"""ARI""",16,13,3,0,489,313,176,0.8125,30.5625,19.5625,11.0
2015,"""ATL""",16,8,8,0,339,345,-6,0.5,21.1875,21.5625,-0.375
2015,"""BAL""",16,5,11,0,328,401,-73,0.3125,20.5,25.0625,-4.5625
2015,"""BUF""",16,8,8,0,379,359,20,0.5,23.6875,22.4375,1.25
2015,"""CAR""",16,15,1,0,500,308,192,0.9375,31.25,19.25,12.0
2015,"""CHI""",16,6,10,0,335,397,-62,0.375,20.9375,24.8125,-3.875
2015,"""CIN""",16,12,4,0,419,279,140,0.75,26.1875,17.4375,8.75
2015,"""CLE""",16,3,13,0,278,432,-154,0.1875,17.375,27.0,-9.625
2015,"""DAL""",16,4,12,0,275,374,-99,0.25,17.1875,23.375,-6.1875


In [9]:
team_season = team_season.with_columns([
    (
        pl.col("wins") - pl.col("losses")
    ).alias("win_loss_diff"),

    (
        pl.col("point_diff") > 0
    ).cast(pl.Int8).alias("positive_point_diff")
])

## One Score Games

A team's record in close games can make its overall record look better or worse than its underlying performance. I want to track one score games separately so I can later test how much that performance carries over from year to year.

In [10]:
team_games = team_games.with_columns(
    (pl.col("point_diff").abs() <= 8)
    .cast(pl.Int8)
    .alias("one_score_game")
)

one_score_summary = (
    team_games
    .filter(pl.col("one_score_game") == 1)
    .group_by([
        "season",
        "team"
    ])
    .agg([
        pl.len().alias("one_score_games"),
        pl.col("win").sum().alias("one_score_wins"),
        pl.col("loss").sum().alias("one_score_losses"),
        pl.col("tie").sum().alias("one_score_ties")
    ])
    .with_columns(
        (
            (
                pl.col("one_score_wins") +
                0.5 * pl.col("one_score_ties")
            ) /
            pl.col("one_score_games")
        ).alias("one_score_win_pct")
    )
)

In [11]:
team_season = (
    team_season
    .join(
        one_score_summary,
        on=[
            "season",
            "team"
        ],
        how="left"
    )
    .with_columns([
        pl.col("one_score_games").fill_null(0),
        pl.col("one_score_wins").fill_null(0),
        pl.col("one_score_losses").fill_null(0),
        pl.col("one_score_ties").fill_null(0),
        pl.col("one_score_win_pct").fill_null(0)
    ])
)

## Play-by-Play Efficiency

Final scores can hide a lot of what actually happened during a game, so I also want to measure how efficiently each team played on a snap-by-snap basis.

EPA and success rate give us a better idea of whether an offense or defense was consistently creating value rather than just looking at the final score.

In [13]:
pbp = pl.read_parquet(
    PROCESSED_DIR / "play_by_play_clean.parquet"
)

print(pbp.shape)
print(pbp.columns)

(508914, 45)
['game_id', 'play_id', 'season', 'week', 'home_team', 'away_team', 'posteam', 'defteam', 'qtr', 'down', 'ydstogo', 'yardline_100', 'goal_to_go', 'score_differential', 'play_type', 'yards_gained', 'epa', 'success', 'wp', 'wpa', 'pass_attempt', 'rush_attempt', 'qb_dropback', 'qb_scramble', 'sack', 'qb_hit', 'complete_pass', 'interception', 'fumble', 'fumble_lost', 'touchdown', 'pass_touchdown', 'rush_touchdown', 'air_yards', 'yards_after_catch', 'cpoe', 'xpass', 'pass_oe', 'first_down', 'third_down_converted', 'third_down_failed', 'fourth_down_converted', 'fourth_down_failed', 'shotgun', 'no_huddle']


In [14]:
offensive_plays = (
    pbp
    .filter(
        pl.col("posteam").is_not_null() &
        pl.col("defteam").is_not_null() &
        pl.col("epa").is_not_null() &
        (
            (pl.col("pass_attempt") == 1) |
            (pl.col("rush_attempt") == 1)
        )
    )
)

offensive_plays.shape

(367199, 45)

In [16]:
offensive_plays = (
    pbp
    .filter(
        pl.col("posteam").is_not_null() &
        pl.col("defteam").is_not_null() &
        pl.col("epa").is_not_null() &
        pl.col("play_type").is_in([
            "pass",
            "run"
        ])
    )
)

In [17]:
offensive_efficiency = (
    offensive_plays
    .group_by([
        "season",
        "posteam"
    ])
    .agg([
        pl.len().alias("off_plays"),
        pl.col("epa").mean().alias("off_epa_per_play"),
        pl.col("success").mean().alias("off_success_rate"),

        pl.col("epa")
        .filter(pl.col("play_type") == "pass")
        .mean()
        .alias("off_pass_epa_per_play"),

        pl.col("epa")
        .filter(pl.col("play_type") == "run")
        .mean()
        .alias("off_rush_epa_per_play")
    ])
    .rename({
        "posteam": "team"
    })
)

In [18]:
defensive_efficiency = (
    offensive_plays
    .group_by([
        "season",
        "defteam"
    ])
    .agg([
        pl.len().alias("def_plays"),
        pl.col("epa").mean().alias("def_epa_per_play_allowed"),
        pl.col("success").mean().alias("def_success_rate_allowed"),

        pl.col("epa")
        .filter(pl.col("play_type") == "pass")
        .mean()
        .alias("def_pass_epa_per_play_allowed"),

        pl.col("epa")
        .filter(pl.col("play_type") == "run")
        .mean()
        .alias("def_rush_epa_per_play_allowed")
    ])
    .rename({
        "defteam": "team"
    })
)

In [19]:
team_season = (
    team_season
    .join(
        offensive_efficiency,
        on=[
            "season",
            "team"
        ],
        how="left"
    )
    .join(
        defensive_efficiency,
        on=[
            "season",
            "team"
        ],
        how="left"
    )
)

## Explosive Plays

EPA measures overall efficiency, but it is also useful to separate teams that consistently create big plays.

I define an explosive play as a pass gaining at least 20 yards or a run gaining at least 10 yards.

In [21]:
explosive_plays = (
    offensive_plays
    .with_columns(
        (
            ((pl.col("play_type") == "pass") & (pl.col("yards_gained") >= 20)) |
            ((pl.col("play_type") == "run") & (pl.col("yards_gained") >= 10))
        )
        .cast(pl.Int8)
        .alias("explosive")
    )
)

In [22]:
offensive_explosiveness = (
    explosive_plays
    .group_by([
        "season",
        "posteam"
    ])
    .agg([
        pl.col("explosive").mean().alias("off_explosive_rate"),

        pl.col("explosive")
        .filter(pl.col("play_type") == "pass")
        .mean()
        .alias("off_explosive_pass_rate"),

        pl.col("explosive")
        .filter(pl.col("play_type") == "run")
        .mean()
        .alias("off_explosive_rush_rate")
    ])
    .rename({
        "posteam": "team"
    })
)

In [23]:
defensive_explosiveness = (
    explosive_plays
    .group_by([
        "season",
        "defteam"
    ])
    .agg([
        pl.col("explosive").mean().alias("def_explosive_rate_allowed"),

        pl.col("explosive")
        .filter(pl.col("play_type") == "pass")
        .mean()
        .alias("def_explosive_pass_rate_allowed"),

        pl.col("explosive")
        .filter(pl.col("play_type") == "run")
        .mean()
        .alias("def_explosive_rush_rate_allowed")
    ])
    .rename({
        "defteam": "team"
    })
)

In [24]:
team_season = (
    team_season
    .join(
        offensive_explosiveness,
        on=["season", "team"],
        how="left"
    )
    .join(
        defensive_explosiveness,
        on=["season", "team"],
        how="left"
    )
)

In [25]:
team_season.select([
    "season",
    "team",
    "off_explosive_rate",
    "def_explosive_rate_allowed"
]).filter(
    pl.col("season") == 2025
).sort(
    "off_explosive_rate",
    descending=True
)

season,team,off_explosive_rate,def_explosive_rate_allowed
i32,str,f64,f64
2025,"""BUF""",0.121127,0.108421
2025,"""NE""",0.120944,0.086957
2025,"""BAL""",0.119247,0.103064
2025,"""LA""",0.118371,0.086142
2025,"""CHI""",0.115876,0.117417
…,…,…,…
2025,"""NYJ""",0.078295,0.115859
2025,"""ARI""",0.078212,0.108262
2025,"""CLE""",0.07393,0.089537


## Turnovers and Pressure

Turnovers can swing games quickly, but they are also one of the more volatile parts of team performance. I want to track them separately instead of letting them get mixed into the rest of the efficiency metrics.

Pressure related stats can also help show whether a defense is consistently affecting the quarterback even when it does not always result in a sack.

In [26]:
offensive_turnovers = (
    offensive_plays
    .group_by([
        "season",
        "posteam"
    ])
    .agg([
        pl.col("interception").sum().alias("interceptions_thrown"),
        pl.col("fumble_lost").sum().alias("fumbles_lost"),
        (
            pl.col("interception").sum() +
            pl.col("fumble_lost").sum()
        ).alias("turnovers")
    ])
    .rename({
        "posteam": "team"
    })
)

In [27]:
defensive_turnovers = (
    offensive_plays
    .group_by([
        "season",
        "defteam"
    ])
    .agg([
        pl.col("interception").sum().alias("def_interceptions"),
        pl.col("fumble_lost").sum().alias("def_fumble_recoveries"),
        (
            pl.col("interception").sum() +
            pl.col("fumble_lost").sum()
        ).alias("takeaways")
    ])
    .rename({
        "defteam": "team"
    })
)

In [28]:
defensive_pressure = (
    offensive_plays
    .filter(pl.col("play_type") == "pass")
    .group_by([
        "season",
        "defteam"
    ])
    .agg([
        pl.len().alias("def_pass_plays"),
        pl.col("sack").sum().alias("sacks"),
        pl.col("qb_hit").sum().alias("qb_hits"),
        pl.col("sack").mean().alias("sack_rate"),
        pl.col("qb_hit").mean().alias("qb_hit_rate")
    ])
    .rename({
        "defteam": "team"
    })
)

In [29]:
team_season = (
    team_season
    .join(
        offensive_turnovers,
        on=["season", "team"],
        how="left"
    )
    .join(
        defensive_turnovers,
        on=["season", "team"],
        how="left"
    )
    .join(
        defensive_pressure,
        on=["season", "team"],
        how="left"
    )
    .with_columns(
        (pl.col("takeaways") - pl.col("turnovers"))
        .alias("turnover_margin")
    )
)

## Situational Efficiency

Overall efficiency is important, but certain situations can have a bigger effect on games.

I want to track how teams perform on third down, fourth down, and near the goal line so those situations can be evaluated separately from overall play-by-play efficiency.

In [31]:
offensive_situational = (
    pbp
    .filter(
        pl.col("posteam").is_not_null() &
        pl.col("defteam").is_not_null()
    )
    .group_by([
        "season",
        "posteam"
    ])
    .agg([
        pl.col("third_down_converted").sum().alias("third_down_conversions"),
        (
            pl.col("third_down_converted").sum() +
            pl.col("third_down_failed").sum()
        ).alias("third_down_attempts"),

        pl.col("fourth_down_converted").sum().alias("fourth_down_conversions"),
        (
            pl.col("fourth_down_converted").sum() +
            pl.col("fourth_down_failed").sum()
        ).alias("fourth_down_attempts")
    ])
    .with_columns([
        (
            pl.col("third_down_conversions") /
            pl.col("third_down_attempts")
        ).alias("off_third_down_rate"),

        (
            pl.col("fourth_down_conversions") /
            pl.col("fourth_down_attempts")
        ).alias("off_fourth_down_rate")
    ])
    .rename({
        "posteam": "team"
    })
)

In [32]:
defensive_situational = (
    pbp
    .filter(
        pl.col("posteam").is_not_null() &
        pl.col("defteam").is_not_null()
    )
    .group_by([
        "season",
        "defteam"
    ])
    .agg([
        pl.col("third_down_converted").sum().alias("third_down_conversions_allowed"),
        (
            pl.col("third_down_converted").sum() +
            pl.col("third_down_failed").sum()
        ).alias("third_down_attempts_faced"),

        pl.col("fourth_down_converted").sum().alias("fourth_down_conversions_allowed"),
        (
            pl.col("fourth_down_converted").sum() +
            pl.col("fourth_down_failed").sum()
        ).alias("fourth_down_attempts_faced")
    ])
    .with_columns([
        (
            pl.col("third_down_conversions_allowed") /
            pl.col("third_down_attempts_faced")
        ).alias("def_third_down_rate_allowed"),

        (
            pl.col("fourth_down_conversions_allowed") /
            pl.col("fourth_down_attempts_faced")
        ).alias("def_fourth_down_rate_allowed")
    ])
    .rename({
        "defteam": "team"
    })
)

In [33]:
team_season = (
    team_season
    .join(
        offensive_situational,
        on=["season", "team"],
        how="left"
    )
    .join(
        defensive_situational,
        on=["season", "team"],
        how="left"
    )
)

In [35]:
red_zone_plays = (
    offensive_plays
    .filter(
        pl.col("yardline_100") <= 20
    )
)

In [36]:
offensive_red_zone = (
    red_zone_plays
    .group_by([
        "season",
        "posteam"
    ])
    .agg([
        pl.len().alias("off_red_zone_plays"),
        pl.col("epa").mean().alias("off_red_zone_epa_per_play"),
        pl.col("success").mean().alias("off_red_zone_success_rate")
    ])
    .rename({
        "posteam": "team"
    })
)

defensive_red_zone = (
    red_zone_plays
    .group_by([
        "season",
        "defteam"
    ])
    .agg([
        pl.len().alias("def_red_zone_plays"),
        pl.col("epa").mean().alias("def_red_zone_epa_per_play_allowed"),
        pl.col("success").mean().alias("def_red_zone_success_rate_allowed")
    ])
    .rename({
        "defteam": "team"
    })
)

In [37]:
team_season = (
    team_season
    .join(
        offensive_red_zone,
        on=["season", "team"],
        how="left"
    )
    .join(
        defensive_red_zone,
        on=["season", "team"],
        how="left"
    )
)

In [38]:
team_season.select([
    "season",
    "team",
    "off_red_zone_plays",
    "off_red_zone_epa_per_play",
    "off_red_zone_success_rate",
    "def_red_zone_epa_per_play_allowed"
]).filter(
    pl.col("season") == 2025
).sort(
    "off_red_zone_epa_per_play",
    descending=True
)

season,team,off_red_zone_plays,off_red_zone_epa_per_play,off_red_zone_success_rate,def_red_zone_epa_per_play_allowed
i32,str,u32,f64,f64,f64
2025,"""SF""",201,0.186114,0.462687,-0.075591
2025,"""IND""",194,0.158664,0.453608,-0.096994
2025,"""DEN""",170,0.149269,0.447059,-0.155656
2025,"""DET""",191,0.149179,0.408377,-0.050149
2025,"""BUF""",199,0.103244,0.482412,0.103826
…,…,…,…,…,…
2025,"""NO""",126,-0.143435,0.357143,-0.061601
2025,"""BAL""",177,-0.150634,0.372881,-0.186504
2025,"""NYJ""",129,-0.153498,0.418605,0.243435


# Roster Continuity

Team performance can change quickly when a large part of the roster turns over.

I want to measure how much playing time returns from the previous season, both overall and by position group. This should give the model a better idea of which teams are keeping the same core together and which teams are replacing a large number of contributors.

In [39]:
snap_counts = pl.read_parquet(
    PROCESSED_DIR / "snap_counts_clean.parquet"
)

print(snap_counts.shape)
print(snap_counts.columns)

(264774, 14)
['game_id', 'season', 'week', 'player', 'pfr_player_id', 'position', 'team', 'opponent', 'offense_snaps', 'offense_pct', 'defense_snaps', 'defense_pct', 'st_snaps', 'st_pct']


In [40]:
player_snaps = (
    snap_counts
    .group_by([
        "season",
        "team",
        "pfr_player_id",
        "player",
        "position"
    ])
    .agg([
        pl.col("offense_snaps").sum().alias("offense_snaps"),
        pl.col("defense_snaps").sum().alias("defense_snaps")
    ])
    .with_columns(
        (
            pl.col("offense_snaps") +
            pl.col("defense_snaps")
        ).alias("total_snaps")
    )
)

In [41]:
previous_snaps = (
    player_snaps
    .select([
        pl.col("season") + 1,
        "team",
        "pfr_player_id",
        pl.col("offense_snaps").alias("previous_offense_snaps"),
        pl.col("defense_snaps").alias("previous_defense_snaps"),
        pl.col("total_snaps").alias("previous_total_snaps")
    ])
)

In [42]:
returning_players = (
    player_snaps
    .join(
        previous_snaps,
        on=[
            "season",
            "team",
            "pfr_player_id"
        ],
        how="left"
    )
    .with_columns([
        pl.col("previous_offense_snaps").fill_null(0),
        pl.col("previous_defense_snaps").fill_null(0),
        pl.col("previous_total_snaps").fill_null(0)
    ])
)

In [43]:
roster_continuity = (
    returning_players
    .filter(pl.col("season") >= 2016)
    .group_by([
        "season",
        "team"
    ])
    .agg([
        pl.col("previous_offense_snaps").sum().alias("returning_offense_snaps"),
        pl.col("previous_defense_snaps").sum().alias("returning_defense_snaps"),
        pl.col("previous_total_snaps").sum().alias("returning_total_snaps")
    ])
)

In [44]:
previous_team_snaps = (
    player_snaps
    .group_by([
        "season",
        "team"
    ])
    .agg([
        pl.col("offense_snaps").sum().alias("team_offense_snaps"),
        pl.col("defense_snaps").sum().alias("team_defense_snaps"),
        pl.col("total_snaps").sum().alias("team_total_snaps")
    ])
    .with_columns(
        pl.col("season") + 1
    )
)

In [46]:
roster_continuity = (
    roster_continuity
    .join(
        previous_team_snaps,
        on=[
            "season",
            "team"
        ],
        how="left"
    )
    .with_columns([
        (
            pl.col("returning_offense_snaps") /
            pl.col("team_offense_snaps")
        ).alias("offense_continuity"),

        (
            pl.col("returning_defense_snaps") /
            pl.col("team_defense_snaps")
        ).alias("defense_continuity"),

        (
            pl.col("returning_total_snaps") /
            pl.col("team_total_snaps")
        ).alias("overall_continuity")
    ])
)

In [47]:
team_season = (
    team_season
    .join(
        roster_continuity.select([
            "season",
            "team",
            "offense_continuity",
            "defense_continuity",
            "overall_continuity"
        ]),
        on=[
            "season",
            "team"
        ],
        how="left"
    )
)

### Position Group Continuity

Overall continuity does not capture where roster turnover happened. Returning players at quarterback or along the offensive line can mean something different than returning the same amount of playing time at other positions.

I separate a few major position groups so the model can account for where teams experienced the most change.

In [52]:
player_snaps = (
    player_snaps
    .with_columns(
        pl.when(pl.col("position") == "QB")
        .then(pl.lit("QB"))

        .when(pl.col("position").is_in([
            "C", "C/G", "G", "G/C", "G/OT", "G/T",
            "OG", "OL", "OT", "T", "T/G"
        ]))
        .then(pl.lit("OL"))

        .when(pl.col("position").is_in([
            "RB", "RB/F", "RB/W", "HB",
            "FB", "FB/D", "FB/R", "FB/T",
            "WR", "WR/R", "TE", "TE/D"
        ]))
        .then(pl.lit("Skill"))

        .when(pl.col("position").is_in([
            "DE", "DE/D", "DE/L", "DL", "DT", "DT/D",
            "NT", "LB", "LB/F", "ILB", "MLB", "OLB"
        ]))
        .then(pl.lit("Front Seven"))

        .when(pl.col("position").is_in([
            "CB", "CB/R", "DB", "DB/L",
            "FS", "S", "SS"
        ]))
        .then(pl.lit("Secondary"))

        .otherwise(None)
        .alias("position_group")
    )
)

In [58]:
position_group_snaps = (
    player_snaps
    .filter(
        pl.col("position_group").is_not_null()
    )
    .group_by([
        "season",
        "team",
        "pfr_player_id",
        "position_group"
    ])
    .agg(
        pl.col("total_snaps")
        .sum()
        .alias("total_snaps")
    )
)

In [64]:
previous_position_snaps = (
    position_group_snaps
    .select([
        pl.col("season") + 1,
        "team",
        "pfr_player_id",
        "position_group",
        pl.col("total_snaps").alias("previous_position_snaps")
    ])
)

In [69]:
returning_position_snaps = (
    position_group_snaps
    .join(
        previous_position_snaps,
        on=[
            "season",
            "team",
            "pfr_player_id",
            "position_group"
        ],
        how="left"
    )
    .with_columns(
        pl.col("previous_position_snaps").fill_null(0)
    )
)

In [73]:
previous_position_totals = (
    position_group_snaps
    .group_by([
        "season",
        "team",
        "position_group"
    ])
    .agg(
        pl.col("total_snaps")
        .sum()
        .alias("position_snaps")
    )
    .with_columns(
        pl.col("season") + 1
    )
)

In [76]:
position_continuity = (
    returning_position_snaps
    .filter(pl.col("season") >= 2016)
    .group_by([
        "season",
        "team",
        "position_group"
    ])
    .agg(
        pl.col("previous_position_snaps")
        .sum()
        .alias("returning_position_snaps")
    )
    .join(
        previous_position_totals,
        on=[
            "season",
            "team",
            "position_group"
        ],
        how="left"
    )
    .with_columns(
        (
            pl.col("returning_position_snaps") /
            pl.col("position_snaps")
        ).alias("continuity")
    )
)

In [79]:
position_continuity_wide = (
    position_continuity
    .select([
        "season",
        "team",
        "position_group",
        "continuity"
    ])
    .pivot(
        values="continuity",
        index=["season", "team"],
        on="position_group"
    )
    .rename({
        "QB": "qb_continuity",
        "OL": "ol_continuity",
        "Skill": "skill_continuity",
        "Front Seven": "front_seven_continuity",
        "Secondary": "secondary_continuity"
    })
)

In [80]:
team_season = (
    team_season
    .join(
        position_continuity_wide,
        on=["season", "team"],
        how="left"
    )
)

In [81]:
team_season.select([
    "season",
    "team",
    "qb_continuity",
    "ol_continuity",
    "skill_continuity",
    "front_seven_continuity",
    "secondary_continuity"
]).filter(
    pl.col("season") == 2025
).head(10)

season,team,qb_continuity,ol_continuity,skill_continuity,front_seven_continuity,secondary_continuity
i32,str,f64,f64,f64,f64,f64
2025,"""ARI""",0.971715,0.895176,0.971722,0.512751,0.706805
2025,"""ATL""",1.0,0.696641,0.976549,0.492808,0.75947
2025,"""BAL""",0.967568,0.813037,0.913524,0.860389,0.592425
2025,"""BUF""",0.981651,0.987496,0.735756,0.732124,0.79177
2025,"""CAR""",1.0,0.999809,0.678102,0.438638,0.566495
2025,"""CHI""",1.0,0.324149,0.726245,0.678518,0.955373
2025,"""CIN""",1.0,0.57914,0.831169,0.57506,0.736045
2025,"""CLE""",0.0,0.787584,0.512434,0.551913,0.62069
2025,"""DAL""",0.427839,0.817096,0.753717,0.442504,0.709029


# Player Production

Roster continuity tells me who came back, but it does not tell me how productive those players were.

I want to summarize player production by team and season so I can later measure how much offensive value, usage, and experience each roster is carrying forward.

In [82]:
player_stats = pl.read_parquet(
    PROCESSED_DIR / "player_stats_clean.parquet"
)

print(player_stats.shape)
print(player_stats.columns)

(21366, 148)
['player_id', 'player_name', 'player_display_name', 'position', 'position_group', 'headshot_url', 'season', 'season_type', 'recent_team', 'games', 'completions', 'attempts', 'passing_yards', 'passing_tds', 'passing_interceptions', 'sacks_suffered', 'sack_yards_lost', 'sack_fumbles', 'sack_fumbles_lost', 'passing_air_yards', 'passing_yards_after_catch', 'passing_first_downs', 'passing_epa', 'passing_cpoe', 'passing_2pt_conversions', 'pacr', 'passing_10', 'passing_16', 'passing_20', 'passing_40', 'carries', 'rushing_yards', 'rushing_tds', 'rushing_fumbles', 'rushing_fumbles_lost', 'rushing_first_downs', 'rushing_epa', 'rushing_2pt_conversions', 'rushing_10', 'rushing_12', 'rushing_20', 'rushing_40', 'receptions', 'targets', 'receiving_yards', 'receiving_tds', 'receiving_fumbles', 'receiving_fumbles_lost', 'receiving_air_yards', 'receiving_yards_after_catch', 'receiving_first_downs', 'receiving_epa', 'receiving_2pt_conversions', 'receiving_10', 'receiving_16', 'receiving_20',

In [83]:
player_production = (
    player_stats
    .select([
        "player_id",
        "player_name",
        "player_display_name",
        "position",
        "position_group",
        "season",
        pl.col("recent_team").alias("team"),
        "games",

        "attempts",
        "completions",
        "passing_yards",
        "passing_tds",
        "passing_interceptions",
        "sacks_suffered",
        "passing_epa",
        "passing_cpoe",

        "carries",
        "rushing_yards",
        "rushing_tds",
        "rushing_fumbles_lost",
        "rushing_epa",

        "receptions",
        "targets",
        "receiving_yards",
        "receiving_tds",
        "receiving_fumbles_lost",
        "receiving_epa",
        "target_share",
        "air_yards_share",

        "def_tackles_solo",
        "def_tackle_assists",
        "def_tackles_for_loss",
        "def_fumbles_forced",
        "def_sacks",
        "def_qb_hits",
        "def_interceptions",
        "def_pass_defended"
    ])
    .sort([
        "season",
        "team",
        "player_id"
    ])
)

In [84]:
player_production = (
    player_production
    .with_columns([
        (pl.col("passing_yards") / pl.col("games"))
        .alias("passing_yards_per_game"),

        (pl.col("rushing_yards") / pl.col("games"))
        .alias("rushing_yards_per_game"),

        (pl.col("receiving_yards") / pl.col("games"))
        .alias("receiving_yards_per_game"),

        (pl.col("targets") / pl.col("games"))
        .alias("targets_per_game"),

        pl.when(pl.col("attempts") > 0)
        .then(pl.col("completions") / pl.col("attempts"))
        .otherwise(None)
        .alias("completion_rate"),

        pl.when(pl.col("attempts") > 0)
        .then(pl.col("passing_interceptions") / pl.col("attempts"))
        .otherwise(None)
        .alias("interception_rate"),

        pl.when(pl.col("carries") > 0)
        .then(pl.col("rushing_yards") / pl.col("carries"))
        .otherwise(None)
        .alias("yards_per_carry"),

        pl.when(pl.col("targets") > 0)
        .then(pl.col("receptions") / pl.col("targets"))
        .otherwise(None)
        .alias("catch_rate"),

        pl.when(pl.col("targets") > 0)
        .then(pl.col("receiving_yards") / pl.col("targets"))
        .otherwise(None)
        .alias("yards_per_target")
    ])
)

In [85]:
print("Rows:", player_production.height)

print(
    "Duplicate player-seasons:",
    player_production
    .group_by([
        "season",
        "player_id"
    ])
    .len()
    .filter(pl.col("len") > 1)
    .height
)

print(
    "Infinite rate values:",
    player_production
    .select([
        pl.col("completion_rate").is_infinite().sum(),
        pl.col("interception_rate").is_infinite().sum(),
        pl.col("yards_per_carry").is_infinite().sum(),
        pl.col("catch_rate").is_infinite().sum(),
        pl.col("yards_per_target").is_infinite().sum()
    ])
)

Rows: 21366
Duplicate player-seasons: 0
Infinite rate values: shape: (1, 5)
┌─────────────────┬───────────────────┬─────────────────┬────────────┬──────────────────┐
│ completion_rate ┆ interception_rate ┆ yards_per_carry ┆ catch_rate ┆ yards_per_target │
│ ---             ┆ ---               ┆ ---             ┆ ---        ┆ ---              │
│ u32             ┆ u32               ┆ u32             ┆ u32        ┆ u32              │
╞═════════════════╪═══════════════════╪═════════════════╪════════════╪══════════════════╡
│ 0               ┆ 0                 ┆ 0               ┆ 0          ┆ 0                │
└─────────────────┴───────────────────┴─────────────────┴────────────┴──────────────────┘
